# Applied Cryptography

Authenticated encryption with associated data (AEAD) in Python using AES-GCM and ChaCha20-Poly1305.


### Overview
Demonstrates authenticated encryption with associated data (AEAD):
* Sender builds a 96-bit nonce using a counter and encrypts the payload
* Associated data (AAD) binds the packet sequence to block tampering
* Receiver checks freshness and drops modified or replayed messages
* File encryption to encrypt and restore files on disk


### Setup
Run this cell if you need to install the cryptography library.


In [ ]:
%pip install cryptography


In [ ]:
import os
import time
import struct
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives.ciphers.aead import AESGCM, ChaCha20Poly1305

KEY_SIZE = 32
NONCE_SIZE = 12
TAG_SIZE = 16


### Sender & Receiver
Sender packs the nonce and encrypts; Receiver checks freshness and decrypts.


In [ ]:
class Sender:
    def __init__(self, key: bytes, algo: str = "AES_GCM"):
        if len(key) != KEY_SIZE:
            raise ValueError(f"key must be exactly {KEY_SIZE} bytes")
        self.key = key
        self.algo = algo
        self.seq = 0
        self._cipher = AESGCM(key) if algo == "AES_GCM" else ChaCha20Poly1305(key)

    def protect(self, pt: bytes, aad: bytes = b"") -> dict:
        # 64-bit counter + 4 zero bytes = 96-bit nonce (standard for GCM/Poly1305)
        nonce = struct.pack(">Q", self.seq) + b"\x00\x00\x00\x00"
        full_aad = struct.pack(">Q", self.seq) + aad
        self.seq += 1

        raw = self._cipher.encrypt(nonce, pt, full_aad)
        ct, tag = raw[:-TAG_SIZE], raw[-TAG_SIZE:]

        return {
            "nonce": nonce,
            "aad": full_aad,
            "ct": ct,
            "tag": tag
        }

class Receiver:
    def __init__(self, key: bytes, algo: str = "AES_GCM"):
        if len(key) != KEY_SIZE:
            raise ValueError(f"key must be exactly {KEY_SIZE} bytes")
        self.key = key
        self.algo = algo
        # TODO: replace with sliding window if this runs in long sessions
        self.seen = set()
        self._cipher = AESGCM(key) if algo == "AES_GCM" else ChaCha20Poly1305(key)

    def open(self, record: dict):
        nonce = record["nonce"]
        aad = record["aad"]
        ct = record["ct"]
        tag = record["tag"]

        if len(aad) < 8:
            return None

        seq = struct.unpack(">Q", aad[:8])[0]
        if seq in self.seen:
            return None

        try:
            pt = self._cipher.decrypt(nonce, ct + tag, aad)
            self.seen.add(seq)
            return pt
        except InvalidTag:
            return None


### Security Tests
Checks that tampered ciphertexts, bad tags, modified headers, and replayed packets get rejected.


In [ ]:
for algo in ["AES_GCM", "ChaCha20_Poly1305"]:
    key = os.urandom(32)
    s = Sender(key, algo)
    r = Receiver(key, algo)
    msg = b"Confidential record content"

    rec = s.protect(msg, b"hdr_v1")
    assert r.open(rec) == msg
    print(f"[{algo}] decrypt: PASS")

    bad = dict(rec)
    bad["ct"] = bytes([bad["ct"][0] ^ 0x01]) + bad["ct"][1:]
    assert r.open(bad) is None
    print(f"[{algo}] tampered ct: PASS")

    bad = dict(rec)
    bad["tag"] = bytes([bad["tag"][0] ^ 0xFF]) + bad["tag"][1:]
    assert r.open(bad) is None
    print(f"[{algo}] tampered tag: PASS")

    bad = dict(rec)
    bad["aad"] = bad["aad"][:-1] + b"X"
    assert r.open(bad) is None
    print(f"[{algo}] tampered aad: PASS")

    assert r.open(rec) is None
    print(f"[{algo}] replay blocked: PASS")

    r_bad = Receiver(os.urandom(32), algo)
    fresh = s.protect(b"Another message", b"hdr")
    assert r_bad.open(fresh) is None
    print(f"[{algo}] wrong key: PASS")


### Nonce Uniqueness
Verifies that 10,000 sequential nonces contain zero repeats.


In [ ]:
for algo in ["AES_GCM", "ChaCha20_Poly1305"]:
    s = Sender(os.urandom(32), algo)
    seen = set()
    for _ in range(10000):
        rec = s.protect(b"ping", b"h")
        seen.add(rec["nonce"])
    assert len(seen) == 10000
    print(f"[{algo}] 10k nonces, 0 collisions: PASS")


### Speed Benchmark
Measures encryption and decryption speed across 64 byte, 1 KB, and 64 KB payloads.


In [ ]:
print(f"{'Algorithm':<18} | {'Payload':<10} | {'Enc Time':<10} | {'Dec Time':<10} | {'Speed':<12}")
print("-" * 70)

for algo in ["AES_GCM", "ChaCha20_Poly1305"]:
    key = os.urandom(32)
    s = Sender(key, algo)
    r = Receiver(key, algo)

    for size in [64, 1024, 65536]:
        payload = os.urandom(size)

        t0 = time.time()
        packets = [s.protect(payload, b"meta") for _ in range(200)]
        enc_t = time.time() - t0

        t1 = time.time()
        for p in packets:
            r.open(p)
        dec_t = time.time() - t1

        total_mb = (size * 200) / (1024 * 1024)
        mb_s = total_mb / (enc_t + dec_t)
        print(f"{algo:<18} | {size:>5} bytes | {enc_t:>8.4f}s | {dec_t:>8.4f}s | {mb_s:>8.2f} MB/s")


### File Encryption
Encrypts a test file to disk, restores it, and verifies the content matches.


### Encrypt Your Own File
Upload any file, encrypt it with a random key, then decrypt and verify the content is identical.

In [ ]:
import os
from google.colab import files
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]
plaintext = uploaded[filename]

key = os.urandom(32)
s = Sender(key, "AES_GCM")
r = Receiver(key, "AES_GCM")

rec = s.protect(plaintext, filename.encode())
recovered = r.open(rec)

assert recovered == plaintext
print(f"File: {filename}")
print(f"Size: {len(plaintext):,} bytes")
print(f"Key:  {key.hex()}")
print(f"CT:   {len(rec['ct']):,} bytes")
print("Decrypt verified: PASS")

In [ ]:
def encrypt_file(in_path: str, out_path: str, key: bytes, algo: str = "AES_GCM") -> bool:
    with open(in_path, "rb") as f:
        data = f.read()

    sender = Sender(key, algo)
    meta = os.path.basename(in_path).encode("utf-8")
    rec = sender.protect(data, meta)

    algo_id = 0 if algo == "AES_GCM" else 1
    hdr = struct.pack(">B12s16sI", algo_id, rec["nonce"], rec["tag"], len(rec["aad"]))

    with open(out_path, "wb") as f:
        f.write(hdr + rec["aad"] + rec["ct"])
    return True

def decrypt_file(in_path: str, out_path: str, key: bytes) -> bool:
    with open(in_path, "rb") as f:
        raw = f.read()

    if len(raw) < 33:
        return False

    algo_id, nonce, tag, aad_len = struct.unpack(">B12s16sI", raw[:33])
    algo = "AES_GCM" if algo_id == 0 else "ChaCha20_Poly1305"
    aad = raw[33:33 + aad_len]
    ct = raw[33 + aad_len:]

    receiver = Receiver(key, algo)
    pt = receiver.open({"nonce": nonce, "aad": aad, "ct": ct, "tag": tag})
    if pt is None:
        return False

    with open(out_path, "wb") as f:
        f.write(pt)
    return True

with open("test_sample.txt", "w") as f:
    f.write("Confidential financial report sample data.")

k = os.urandom(32)
encrypt_file("test_sample.txt", "test_sample.enc", k, "AES_GCM")
decrypt_file("test_sample.enc", "test_recovered.txt", k)

with open("test_recovered.txt") as f:
    restored = f.read()

assert restored == "Confidential financial report sample data."
print("File encrypted and successfully restored: PASS")

for f in ["test_sample.txt", "test_sample.enc", "test_recovered.txt"]:
    if os.path.exists(f): os.remove(f)
